# 02 — Hillstrom: randomization check and ATE (Day 1)

Notebook 01 described the data. This one asks the causal question: **was the
e-mail assignment random, and how big is the average effect?** Both answers are
prerequisites for uplift modelling and both are baselines the models must beat.

Vocabulary lives in [`docs/concepts.md`](../../docs/concepts.md) — this notebook
introduces each idea briefly and links back rather than re-deriving it.

| Step | Question | Tool |
| --- | --- | --- |
| Arm choice | Which e-mail arm is worth modelling? | observed lift per arm |
| Randomization | Are treated and control comparable? | standardized mean differences |
| Effect size | What is the average effect? | ATE with 95% CI |

## 0. Setup

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
for _ in range(8):
    if (ROOT / "pyproject.toml").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from packages.causal.diagnostics import ate_table, smd_table
from packages.datasets.hillstrom import (
    FEATURES,
    OUTCOMES,
    arm_summary,
    fetch_hillstrom,
    load_hillstrom_raw,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

raw = load_hillstrom_raw()
print("repo root:", ROOT)

## 1. Choose the binary arm

An **arm** is one branch of the experiment; a **binary** experiment keeps one
e-mail arm and drops the other so the estimand is a clean two-group comparison
(see [concepts §1](../../docs/concepts.md)). Hillstrom has two e-mail arms, so
pick the one with the larger observed lift — with a reason, not a hunch.

In [ ]:
arm_summary(raw).round(4)

**Decision: `mens`.** Mens beats Womens on every outcome, on both scales:

| Lift vs control (`No E-Mail`) | Mens | Womens |
| --- | --- | --- |
| `visit` | 18.3% vs 10.6% → **+72%** | 15.1% vs 10.6% → +43% |
| `conversion` | 1.25% vs 0.57% → **+119%** | 0.88% vs 0.57% → +54% |
| `spend` (mean) | $1.42 vs $0.65 → **+118%** | $1.08 vs $0.65 → +65% |

Mens also produces more absolute events (≈266 vs ≈188 conversions), so its
rates are less noisy and its ATE intervals will be tighter. Nothing here
argues for Womens; `ARM = "mens"` is set below — change it only with a
recorded reason.

In [ ]:
ARM = "mens"          # "mens" or "womens"
df = fetch_hillstrom(arm=ARM)
print(df["segment"].value_counts().to_string())
print("\ntreated share:", round(df["treatment"].mean(), 4))
print("rows:", len(df), " conversions:", int(df["conversion"].sum()))
df.head()

### What is the `treatment` column?

The raw CSV has **no `treatment` column**. It only has `segment`, which holds three
text labels. Models need numbers, so `fetch_hillstrom(arm="mens")` builds one:

| `segment` (from the file) | | `treatment` (added by the loader) |
| --- | --- | --- |
| `Mens E-Mail` | → | `1` — got the e-mail |
| `No E-Mail` | → | `0` — got nothing |
| `Womens E-Mail` | → | *(row dropped)* |

In plain words: **`treatment` is a yes/no flag — 1 means "this customer received
the Mens e-mail", 0 means "they received no e-mail".**

It adds no new information. It is just the `segment` label rewritten as a number,
because the SMD table and the ATE need to group rows into "e-mailed" vs
"not e-mailed" and you cannot do arithmetic on the text `"Mens E-Mail"`.

Two consequences worth remembering:

- It exists only for a **binary** arm. With `arm="multi"` there is no `treatment`
  column — three arms cannot collapse into one 0/1 flag.
- Because it is derived from `segment`, it can never be a model feature — that
  would hand the model the answer. It is listed in `LEAKY_COLUMNS`.

## 2. Randomization check — standardized mean differences

Randomization means treatment is assigned independently of how a customer would
respond, so the two groups should look statistically identical *before* the
e-mail. **SMD** measures the gap on each feature in pooled-spread units so
different features are comparable; the convention here flags `|SMD| > 0.1`
([concepts §2](../../docs/concepts.md)).

Categorical features enter one-vs-rest (e.g. `zip_code=Urban`).

In [ ]:
smd = smd_table(df, FEATURES)
print("flagged (|SMD| > 0.1):", int(smd["flagged"].sum()), "of", len(smd))
print("largest |SMD|:", round(smd["abs_smd"].max(), 4))
smd.round(4)

**Finding.** Zero of 11 comparisons flagged; the largest `|SMD|` is 0.014 —
about one seventh of the 0.1 threshold. Treated and control are balanced on
every pre-treatment feature, which is what random assignment predicts.

**Reading.** The pattern is pure noise, not a systematic gap:

- Signs are mixed — 6 positive, 5 negative — and no feature block (spend
  history, channel, geography) leans one way.
- Magnitudes taper smoothly from 0.014 down to 0.001, and the largest is a
  share-type feature (`zip_code=Rural`, 15.2% vs 14.7%): a gap of ~0.5 points
  is exactly what random fluctuation produces at this `n`.
- The customer-profile features (`history`, `recency`, `mens`, `womens`,
  `newbie`) all sit under 0.008 in *both* directions, so the arms agree on
  *who* the customers are, not just on totals.

**Why a fixed 0.1 and not a p-value.** At ~21k per arm the test is
hypersensitive: an SMD of just 0.03 (a 1.5-point share gap, or ~$8 of mean
`history`) is already statistically significant at p ≈ 0.002, yet plainly too
small to matter. With enough rows *every* imbalance becomes "significant", so
a p-value would eventually flag a perfectly good randomization. The fixed 0.1
threshold asks the question that actually matters — *would this gap change a
decision?* — and does not shrink as `n` grows
([concepts §2](../../docs/concepts.md)).

## 3. Average treatment effect (with 95% CI)

The **ATE** is mean(treated) − mean(control) per outcome; the 95% interval says
where the true average effect plausibly sits. Binary outcomes use the
two-proportion z interval, `spend` uses Welch's t interval
([concepts §2](../../docs/concepts.md)).

This is the number **every uplift model must beat**. Uniformly contacting everyone
buys exactly the ATE; random targeting scores ~0.

In [ ]:
ate = ate_table(df, OUTCOMES)
ate.round(4)

**Result.** All three effects are positive and every 95% interval excludes
zero:

| Outcome | ATE | 95% CI | Distinguishable from 0? |
| --- | --- | --- | --- |
| `visit` | **+0.0766** (+7.66 pp) | [+0.0700, +0.0832] | yes (z ≈ 22.6) |
| `conversion` | **+0.0068** (+0.68 pp) | [+0.0050, +0.0086] | yes (z ≈ 7.4) |
| `spend` | **+$0.7698** | [$0.4851, $1.0545] | yes (t ≈ 5.3) |

**Reading, in words.**

- **`visit`:** contacting 1,000 customers like these yields about **77 extra
  website visits** within two weeks (183 vs 106 per 1,000), with the interval
  putting the true figure at 70–83. It is the largest and tightest effect.
- **`visit` moves more than `conversion` — in absolute points** (7.66 pp vs
  0.68 pp), because visit sits upstream in the funnel: every converter must
  first visit (9,394 visited vs 578 bought), so an effect that starts at the
  top shrinks as it passes each step. In *relative* terms the ordering flips —
  conversion more than doubles (+119% vs +72%) — the rarer downstream event
  gains proportionally more. Report both scales; quoting only one misleads.
- **`spend` is real but imprecise.** Its interval [$0.49, $1.05] spans ±37% of
  the estimate, because `spend` is zero for 99.1% of rows (notebook 01 §5):
  all the variance comes from the 389 converters in this binary arm (~266
  treated, ~123 control), so the *effective* sample is ~389 rows, not 42,613.
  Implication for Day 3+: never quote a point profit from `spend` alone —
  prefer the conversion-based formula with an assumed `v`, and show a range
  (README risk table).

These three numbers are the **baseline Day 2 must beat**: contacting everyone
uniformly earns exactly the ATE, and random targeting scores ~0 — a model is
only worth building if it beats both.

## 4. Profit assumptions

The ATE gives incremental *outcomes*; turning them into money needs assumptions.
They live in `configs/business_assumptions.yaml` (the copilot reads that file)
and mirror into the README — never buried in code. Decided on Day 1; change
only with a recorded reason:

| Assumption | Value | Why |
| --- | --- | --- |
| Reporting currency | USD | fixed — Hillstrom `spend` is already USD |
| Gross margin on incremental spend | **30%** | retail gross-margin placeholder — an assumption, not something the data observes |
| Cost per contact (e-mail) | **$0.10** | typical per-send cost — assumption |
| Value per conversion (Criteo only) | **$34.91** | $116.36 mean spend per converter (notebook 01) × 30% margin — puts Criteo's profit on Hillstrom's scale |

Two equivalent formulas, both in
[concepts — profit formula](../../docs/concepts.md):

```text
profit(k) = v · Δconv(k) − c · n(k)         # conversion route — works on any dataset
profit(k) = margin · Δspend(k) − c · n(k)   # spend route — Hillstrom only
```

On Hillstrom the two agree because `v` *was defined as* margin × mean value
per converter; Criteo records no spend, so only the conversion route exists
there. The cell below loads the config and applies both routes to the ATE
from §3.

In [ ]:
import yaml

cfg = yaml.safe_load((ROOT / "configs" / "business_assumptions.yaml").read_text())
asm = cfg["hillstrom"]
margin, cpc, v = asm["margin_rate"], asm["cost_per_contact_usd"], asm["value_per_conversion_usd"]

N = 10_000  # a small campaign: contact 10,000 customers uniformly
d_conv = float(ate.loc[ate["outcome"] == "conversion", "ate"].iloc[0])
d_spend = float(ate.loc[ate["outcome"] == "spend", "ate"].iloc[0])

rows = {
    "conversion route (v · Δconv)": v * d_conv * N,
    "spend route (margin · Δspend)": margin * d_spend * N,
}
profit = pd.DataFrame({"gross profit (USD)": rows})
profit["contact cost (USD)"] = cpc * N
profit["net profit (USD)"] = profit["gross profit (USD)"] - profit["contact cost (USD)"]
profit.round(2)

**Reading.** Both routes say the average mens campaign is profitable at these
assumptions: ≈ **+$0.13–$0.14 per contact** (+$1,309 to +$1,376 on this
10,000-contact campaign, after paying $1,000 in send costs). The routes differ
~5% because `v` was derived from the *overall* mean spend per converter
($116.36) while the incremental converters averaged slightly less
(≈ $113 = Δspend/Δconv) — a reminder that every profit figure inherits the
assumptions' rounding.

Two caveats to carry into Day 2: this is an *average* profit — the whole point
of uplift modelling is finding customers worth more than +$0.13 — and it rests
on the 30% / $0.10 assumptions above, which every brief must restate as
assumptions, never as observed facts.

## 5. Handoff to Day 2

- [x] SMD table reviewed — verdict: balanced, 0 of 11 flagged, largest `|SMD|` 0.014 (§2)
- [x] ATE table interpreted — all positive, all significant: `visit` +7.66 pp,
  `conversion` +0.68 pp, `spend` +$0.77 (§3)
- [x] Binary arm chosen and justified (`mens`, §1)
- [x] Profit assumptions decided and recorded: 30% margin, $0.10/contact,
  `v` = $34.91 — `configs/business_assumptions.yaml` + README (§4)

Then Day 2: a response baseline versus S-/T-/X-learners, evaluated with Qini
curves — the first test of whether modelling heterogeneity beats the ATE
(≈ +$0.13 per uniform contact at the assumptions above).